# Moshi → Bridge → AvatarForcing Unified Streaming Pipeline

**Real-Time Speech-to-Avatar on RunPod GPU**

```
User Audio → Moshi LM (tokens @12.5Hz) → Bridge (wav2vec features @25Hz) → AvatarForcing (video @25fps)
```

In [ ]:
import torch, sys
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    total_gb = p.total_memory / 1_000_000_000
    print(f'  GPU {i}: {p.name} | {total_gb:.1f} GB')
print('Python:', sys.version.split()[0])


In [ ]:
import subprocess, sys
# System deps
subprocess.run('apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1'.split('&&')[0].split(), check=True, capture_output=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','libsndfile1','git'], check=True, capture_output=True)
print('System deps OK')


In [ ]:
import subprocess, sys
pkgs = [
    'moshi==0.2.4', 'sphn>=0.1.4', 'sentencepiece>=0.1.99',
    'transformers>=4.40.0', 'safetensors>=0.4.0', 'huggingface_hub>=0.23.0',
    'omegaconf>=2.3.0', 'einops>=0.7.0', 'torchvision>=0.18.0',
    'torchaudio>=2.3.0', 'soundfile>=0.12.1', 'pyyaml>=6.0',
    'pillow>=10.0', 'decord>=0.6.0', 'websockets>=12.0',
    'lmdb>=1.4.0', 'pandas>=2.0', 'opencv-python-headless>=4.8',
    'pyworld>=0.3.4', 'tqdm>=4.66', 'ipywidgets>=8.0',
]
subprocess.run([sys.executable,'-m','pip','install','-q'] + pkgs, check=True)
print('Python packages installed')


In [ ]:
# Try flash-attn (optional, needs CUDA toolkit match)
import subprocess, sys
for pkg in ['flash-attn>=2.5.0','xformers>=0.0.26']:
    try:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=True, timeout=300)
        print('OK:', pkg)
    except Exception as e:
        print('SKIP (not critical):', pkg, str(e)[:60])


In [ ]:
import sys
from pathlib import Path

# ─── CONFIGURE THESE ─────────────────────────────────────────────────────────
WORKSPACE    = Path('/workspace')
BRIDGE_ROOT  = WORKSPACE / 'Moshi-AvatarForcing-bridge'
MOSHI_ROOT   = BRIDGE_ROOT / 'moshi-inference'
AF_ROOT      = BRIDGE_ROOT / 'AvatarForcing-inference'
BRIDGE_MOD   = BRIDGE_ROOT / 'bridge_module'
UNIFIED      = BRIDGE_ROOT / 'unified_pipeline'
WAN_DIR      = AF_ROOT / 'wan_models'
BRIDGE_CKPT  = BRIDGE_ROOT / 'checkpoints' / 'bridge_best.pt'
BRIDGE_CFG   = BRIDGE_MOD / 'config.yaml'
AF_CKPT      = WAN_DIR / 'avatarforcing_checkpoint.pt'
AF_CFG       = AF_ROOT / 'configs' / 'avatarforcing.yaml'
# ─────────────────────────────────────────────────────────────────────────────

for p in [str(BRIDGE_ROOT), str(MOSHI_ROOT), str(AF_ROOT), str(BRIDGE_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Paths set up:')
for p in [BRIDGE_ROOT, MOSHI_ROOT, AF_ROOT, BRIDGE_MOD]:
    print(' ', p, '✓' if p.exists() else '✗ MISSING')


In [ ]:
from huggingface_hub import snapshot_download

WAN_DIR.mkdir(parents=True, exist_ok=True)

print('Downloading Wan2.1-T2V-1.3B (large, ~14GB)...')
snapshot_download(
    repo_id='Wan-AI/Wan2.1-T2V-1.3B',
    local_dir=str(WAN_DIR / 'Wan2.1-T2V-1.3B'),
    ignore_patterns=['*.md'],
)
print('Wan2.1 OK')

print('Downloading wav2vec2-base-960h...')
snapshot_download(
    repo_id='facebook/wav2vec2-base-960h',
    local_dir=str(WAN_DIR / 'wav2vec2-base-960h'),
)
print('wav2vec2 OK')


In [ ]:
# Upload reference image and input audio
import ipywidgets as widgets
from IPython.display import display

upload_img = widgets.FileUpload(accept='image/*', multiple=False, description='Portrait Image')
upload_aud = widgets.FileUpload(accept='audio/*', multiple=False, description='Input Audio')
display(widgets.VBox([upload_img, upload_aud]))
print('Upload your files above, then run the next cell.')


In [ ]:
from pathlib import Path

UPLOADS = BRIDGE_ROOT / 'uploads'
UPLOADS.mkdir(exist_ok=True)

# Save image
if upload_img.value:
    fname, data = next(iter(upload_img.value.items()))
    IMAGE_PATH = str(UPLOADS / fname)
    Path(IMAGE_PATH).write_bytes(data['content'])
    print('Image saved:', IMAGE_PATH)
else:
    IMAGE_PATH = '/path/to/your/portrait.jpg'  # set manually if not using widget
    print('WARN: no upload — set IMAGE_PATH manually')

# Save audio
if upload_aud.value:
    fname, data = next(iter(upload_aud.value.items()))
    AUDIO_PATH = str(UPLOADS / fname)
    Path(AUDIO_PATH).write_bytes(data['content'])
    print('Audio saved:', AUDIO_PATH)
else:
    AUDIO_PATH = '/path/to/your/audio.wav'  # set manually
    print('WARN: no upload — set AUDIO_PATH manually')

OUTPUT_PATH = str(BRIDGE_ROOT / 'output' / 'avatar_output.mp4')
Path(OUTPUT_PATH).parent.mkdir(exist_ok=True)
print('Output path:', OUTPUT_PATH)


In [ ]:
import logging, torch
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s - %(message)s')

from unified_pipeline.config import (
    PipelineConfig, MoshiConfig, BridgeConfig, AvatarForcingConfig,
    DEFAULT_AVATAR_PROMPT
)

cfg = PipelineConfig(
    moshi=MoshiConfig(
        hf_repo='kyutai/moshiko-pytorch-bf16',
        device='cuda',
        dtype=torch.bfloat16,
        use_sampling=True,
        temp=0.8, temp_text=0.7, top_k=250,
    ),
    bridge=BridgeConfig(
        checkpoint_path=str(BRIDGE_CKPT),
        config_path=str(BRIDGE_CFG),
        device='cuda',
        dtype=torch.bfloat16,
        chunk_tokens=4,      # 4 moshi frames = 320ms per bridge chunk
        upsample_factor=2,   # 12.5Hz tokens x2 = 25Hz audio features
        use_kv_cache=True,
        use_projection=True, # project bridge 768-dim -> 9984-dim
        wav2vec_num_layers=13,
    ),
    avatar=AvatarForcingConfig(
        avatarforcing_root=str(AF_ROOT),
        config_path=str(AF_CFG),
        checkpoint_path=str(AF_CKPT),
        device='cuda',
        dtype=torch.bfloat16,
        num_output_frames=21,
        fps=25,
        prompt=DEFAULT_AVATAR_PROMPT,
        seed=42,
    ),
    token_queue_maxsize=128,
    feature_queue_maxsize=64,
    frame_queue_maxsize=32,
)

print('Config OK')
print('Prompt:', cfg.avatar.prompt[:100], '...')


In [ ]:
import gc, torch
torch.cuda.empty_cache(); gc.collect()

from unified_pipeline.model_loader import load_all_models

print('Loading all models (first run downloads ~20GB, may take 10-15 min)...')
models = load_all_models(cfg)

mimi       = models['mimi']
lm_gen     = models['lm_gen']
bridge     = models['bridge']
projection = models['projection']
av_pipe    = models['pipeline']

alloc = torch.cuda.memory_allocated() / 1e9
res   = torch.cuda.memory_reserved()  / 1e9
print(f'\nAll models loaded!')
print(f'VRAM: {alloc:.2f} GB allocated / {res:.2f} GB reserved')


In [ ]:
from unified_pipeline.async_pipeline import UnifiedPipeline

unified = UnifiedPipeline(cfg, models)
unified.setup_reference_image(IMAGE_PATH)
print('Pipeline ready - reference image encoded')


In [ ]:
import sphn, torch

print('Loading audio:', AUDIO_PATH)
in_pcms, _ = sphn.read(AUDIO_PATH, sample_rate=mimi.sample_rate)
audio_tensor = torch.from_numpy(in_pcms).cuda()[None, 0:1]  # [1,1,N]
duration_s = audio_tensor.shape[-1] / mimi.sample_rate
n_samples  = audio_tensor.shape[-1]
print(f'Audio: {duration_s:.2f}s | {n_samples} samples @ {mimi.sample_rate}Hz')


In [ ]:
import asyncio, time

print('Starting streaming pipeline...')
print(f'Output: {OUTPUT_PATH}')

t0 = time.monotonic()

async def run():
    return await unified.run_to_video(
        audio_tensor,
        output_path=OUTPUT_PATH,
        fps=cfg.avatar.fps,
        max_blocks=64,
    )

# Run async pipeline
try:
    loop = asyncio.get_event_loop()
    if loop.is_running():
        import nest_asyncio; nest_asyncio.apply()
        out_path = loop.run_until_complete(run())
    else:
        out_path = asyncio.run(run())
except Exception:
    import nest_asyncio; nest_asyncio.apply()
    out_path = asyncio.get_event_loop().run_until_complete(run())

elapsed = time.monotonic() - t0
n_frames = unified.avatar_generator._generated_frames
fps_achieved = n_frames / max(elapsed, 0.01)
print(f'Done in {elapsed:.1f}s | {n_frames} frames | {fps_achieved:.1f} fps achieved')
print('Output:', out_path)


In [ ]:
from IPython.display import Video, display
display(Video(OUTPUT_PATH, embed=True, width=640))


In [ ]:
# Streaming frame-by-frame preview (first 5 frames)
import asyncio
from PIL import Image as PILImage
from IPython.display import display as ipy_display
import io

print('Streaming preview (first 5 frames):')

async def preview_frames():
    count = 0
    async for frame in unified.run(audio_tensor, max_blocks=4):
        if count >= 5: break
        pil = PILImage.fromarray(frame.pixels.numpy())
        print(f'Frame {frame.frame_idx} | latency={frame.latency_ms:.0f}ms')
        ipy_display(pil)
        count += 1

loop = asyncio.get_event_loop()
import nest_asyncio; nest_asyncio.apply()
loop.run_until_complete(preview_frames())


In [ ]:
# Latency benchmark
import asyncio, nest_asyncio
nest_asyncio.apply()

print('Running 5s latency benchmark...')

async def bench():
    return await unified.benchmark_latency(audio_seconds=5.0)

results = asyncio.get_event_loop().run_until_complete(bench())
print('Latency results:')
for k, v in results.items():
    print(f'  {k}: {v}')


In [ ]:
# Optional: WebSocket server for browser client
# Uncomment and run to start real-time streaming server
# Expose port 8765 in RunPod pod settings

# import asyncio, nest_asyncio
# nest_asyncio.apply()
# from unified_pipeline.async_pipeline import StreamingPipelineServer
# server = StreamingPipelineServer(unified)
# print('Starting WebSocket server on :8765')
# asyncio.get_event_loop().run_until_complete(server.serve('0.0.0.0', 8765))

print('WebSocket server: uncomment above to enable')


In [ ]:
# Cleanup GPU memory
import gc, torch
del models, unified
gc.collect()
torch.cuda.empty_cache()
alloc = torch.cuda.memory_allocated() / 1e9
print(f'VRAM after cleanup: {alloc:.2f} GB allocated')
